# A Source-Reading Path for Production Kernels

> Parallelism decides where computation runs; Kernels decide how efficiently each device executes it. Production source mixes algorithms, layouts, instructions, and synchronization: TMEM names storage, WGMMA names matrix multiply-accumulate instructions, and warp specialization divides labor among thread groups.
>
> Begin with the **computational contract**: input/output shapes, dtypes, and mathematical goal. Then trace Tiles, HBM-to-on-chip movement, overlap, synchronization, scaling, and accumulation precision.
>
> DeepGEMM demonstrates low-precision GEMM, FlashAttention tiled Softmax, Triton block-level programming, and MoE grouped GEMM plus scatter/gather.

The goal is source-navigation skill: find the entry point and main loop, then map functions and instructions back to algorithm steps.


When reading a production Kernel, first establish its shape, dtype, and mathematical contract. Only then do Tile dimensions, loads, and synchronization acquire meaning. On Hopper, TMA moves Tiles asynchronously from HBM to shared memory, WGMMA performs asynchronous matrix multiplication, and warp specialization assigns producer and consumer roles. This appendix applies one route to DeepGEMM, FlashAttention, and MoE grouped GEMM: entry and shapes → movement and overlap → scaling, precision, and synchronization.


## 1. Why Advanced Kernels Matter

Teaching implementations can be correct yet much slower than production code for three reasons.

First, eager PyTorch delegates hardware choices to libraries and cannot always specialize for a particular shape, alignment, and GPU. Second, FP8 cannot be enabled by a simple cast: its narrow dynamic range requires per-block scaling so each Tile uses representable values. Third, MoE cannot efficiently launch one small GEMM per expert; token counts differ and launch overhead dominates. Grouped GEMM packs all expert operations into one launch.

Accordingly, DeepGEMM focuses on FP8 and block scaling, FlashAttention-3 overlaps asynchronous operations, and grouped GEMM batches variable expert matrices.


## 2. DeepGEMM Source-Reading Guide

[DeepGEMM](https://github.com/deepseek-ai/DeepGEMM) is a Hopper-specific FP8 dense and grouped GEMM library. Its deliberately narrow target enables compact, aggressively specialized CUDA source.

Two designs organize the library:

- a TMA-friendly layout for per-block scale factors;
- FP32 accumulation with BF16 output.

We examine scaling, precision, launch layout, and finally the repository call chain.


### 2.1 TMA Layout for Per-Block Scaling

FP8's narrow range makes one scale for an entire matrix inaccurate. Divide matrices into blocks, commonly 128×128, and assign each block a scale:

$$C_{ij}=\sum_k(A_{ik}s^A_{ik})(B_{kj}s^B_{kj}).$$

Scale tensors must also be loaded efficiently. If their layout does not align with data Tiles, TMA needs scattered gathers. DeepGEMM stores scales contiguously along K and aligns them with K Tiles (`NVTE_LAYOUT_BLOCK_SCALING`), allowing a descriptor with a base pointer and strides. For a 4096×4096 matrix with 128×128 blocks, only 1024 FP32 scales—4 KB—are needed, so accuracy improves with negligible memory overhead.


### 2.2 BF16 Output and FP8 Precision

FP8 Tensor Cores accumulate in FP32, while downstream layers often expect BF16. Conversion policy matters: truncation introduces systematic bias, while stochastic rounding is unbiased in expectation. DeepGEMM therefore uses stochastic BF16 output in its epilogue.

Long K dimensions create another issue: repeatedly adding small products to a large FP32 partial sum loses low bits. Hopper GEMMs can periodically move the current partial sum into an outer FP32 accumulator, clear the inner WGMMA accumulator, and continue. This two-level accumulation pattern preserves more information across long reductions.


### 2.3 Kernel Grid and Block Layout

DeepGEMM uses a 2D grid, $(M/T_M,N/T_N)$, with each CTA producing one $T_M×T_N$ output Tile. A producer warp group issues TMA loads; consumer warps issue WGMMA and accumulate results. Hopper `mbarrier` synchronizes them.

Tile dimensions jointly determine shared-memory Tiles, FP32 accumulator registers, occupancy, and pipeline depth. In a four-stage pipeline, the producer loads a future Tile while consumers compute an earlier one. This overlap is necessary for high Tensor Core utilization.


### 2.4 Source-Reading Route

Read DeepGEMM from the public API toward the device loop:

1. `deep_gemm/__init__.py` and `dispatcher.py`: public functions and shape-based dispatch.
2. `deep_gemm/jit_kernels/gemm.py`: inject constants and compile CUDA templates.
3. `csrc/deep_gemm.cuh` and `utils.cuh`: launch configuration and TMA descriptors.
4. `csrc/includes/kernels/gemm/kernel.cuh`: producer/consumer main loop.
5. `warp_specializer.cuh`: barriers and role assignment.
6. `epilogue.cuh`: scaling and FP32-to-BF16 output.
7. `utils/matmul.cuh`: WGMMA wrappers.

For each file, map a source symbol to one contract item, Tile, data movement stage, synchronization event, or numerical step.


In [ ]:
# Print a checklist for following the DeepGEMM source-reading path
deepgemm_reading_path = [
    ("Entry layer (Python)", [
        ("deep_gemm/__init__.py",            "package entry, exporting top-level functions such as fp8_gemm"),
        ("deep_gemm/dispatcher.py",          "select a JIT template by (M,N,K) shape"),
        ("deep_gemm/jit_kernels/gemm.py",    "inject constants into a CUDA template, then compile with NVRTC"),
    ]),
    ("Host launch (C++/CUDA)", [
        ("csrc/deep_gemm.cuh",   "launch entry: set grid/block and start the kernel"),
        ("csrc/utils.cuh",       "construct CUtensorMap and convert a PyTorch tensor into a TMA descriptor"),
    ]),
    ("device kernel（CUDA）", [
        ("csrc/includes/kernels/gemm/kernel.cuh",          "main kernel with producer and consumer loops"),
        ("csrc/includes/kernels/gemm/warp_specializer.cuh", "warp roles plus mbarrier synchronization"),
        ("csrc/includes/kernels/gemm/epilogue.cuh",        "apply per-block scale and convert FP32 to BF16"),
        ("csrc/includes/utils/matmul.cuh",                 "inline PTX assembly wrapper for WGMMA instructions"),
    ]),
    ("Grouped GEMM (MoE variant)", [
        ("csrc/includes/kernels/grouped_gemm/kernel.cuh",  "each CTA processes one block for one expert"),
        ("deep_gemm/jit_kernels/grouped_gemm.py",          "stride calculation and JIT for variable-length experts"),
    ]),
]

print("DeepGEMM source-reading path")
print("=" * 70)
for layer, files in deepgemm_reading_path:
    print(f"\n[{layer}]")
    for path, desc in files:
        print(f"  {path}")
        print(f"      → {desc}")
print()
print("Key observation: trace the implementation from __init__.py all the way to kernel.cuh;")
print("Each layer does one job: choose a template, inject constants, launch, move data, or compute.")


## 3. FlashAttention-2/3 Implementation Details

The FlashAttention appendix established online Softmax and tiling. Later generations primarily improve hardware utilization rather than the mathematical result. The repository provides CUDA implementations under `csrc/flash_attn/` for production performance and a Triton implementation for experimentation and education.


### 3.1 Two Improvements in FlashAttention-2

FA-2 reduces non-matmul FLOPs and improves work partitioning. Tensor Cores are much faster than element-wise CUDA Core operations, so it moves rescaling out of the inner loop and leaves it dominated by `Q @ K.T` and `P @ V`.

FA-1 parallelized mostly over batch and heads, underutilizing SMs for batch-1 inference. FA-2 also partitions Q sequence blocks across CTAs. Each CTA independently streams K/V for its Q block, producing enough work to fill the GPU. Both changes aim to keep Tensor Cores and SMs busy without changing exact Attention semantics.


### 3.2 Hopper Features in FlashAttention-3

FA-3 overlaps WGMMA matmul on one Tile with CUDA Core Softmax on another. Ping-pong scheduling alternates two accumulator buffers: issue WGMMA for Tile $i$, process Softmax for $i-1$, wait for $i$, then issue $i+1$ and process $i$. This hides much of the non-matmul work.

FA-3 can also use FP8 Tensor Cores for QK and PV while retaining FP32 online-Softmax states and accumulation. The general pattern matches DeepGEMM: low precision for throughput, higher precision for numerically sensitive reduction state.


### 3.3 FlashAttention Source-Reading Route

Start with `flash_attn_func.py` and `flash_attn_varlen_func.py`, then follow the C++ extension entry in `csrc/flash_attn/flash_api.cpp`. Read `flash_fwd_launch.h` for shape/dtype dispatch, `flash_fwd_kernel.h` for the FA-2 loop, and `kernels_utils.h` plus `epilogue_fwd.hpp` for Softmax and normalization. Finally compare the Hopper `flash_fwd_kernel.h`, epilogue, and utilities to locate ping-pong scheduling, WGMMA, and FP8 scaling. Read the Triton version before CUDA if block-level code is still unfamiliar.


In [ ]:
# FlashAttention source-reading path
fa_reading_path = [
    ("Python entry", [
        ("flash_attn/flash_attn_func.py",      "public API for one Attention call"),
        ("flash_attn/flash_attn_varlen_func.py", "variable-length version for MoE and packing"),
    ]),
    ("CUDA implementation shared by FA-2 and FA-3", [
        ("csrc/flash_attn/flash_api.cpp",           "PyTorch extension entry, argument checks, and launch"),
        ("csrc/flash_attn/flash_fwd_kernel.h",      "forward-kernel template"),
        ("csrc/flash_attn/flash_fwd_launch.h",      "select grid/block launch configuration by shape"),
        ("csrc/flash_attn/kernels_utils.h",         "device functions for softmax and rescaling"),
    ]),
    ("FA-3 Hopper-specific", [
        ("csrc/flash_attn/hopper/flash_fwd_kernel.h", "main kernel for ping-pong scheduling"),
        ("csrc/flash_attn/hopper/epilogue_fwd.hpp",   "epilogue for FP8 scale processing"),
        ("csrc/flash_attn/hopper/utils.h",            "TMA descriptors plus mbarrier synchronization"),
    ]),
    ("Triton implementation, teaching-friendly", [
        ("flash_attn/flash_attn_triton.py", "complete Triton version of FA in about 300 lines"),
    ]),
]

print("FlashAttention output (first 3 rows):")
print("=" * 70)
for layer, files in fa_reading_path:
    print(f"\n[{layer}]")
    for path, desc in files:
        print(f"  {path}")
        print(f"      → {desc}")
print()
print("Key observation: build intuition with the Triton version, then compare the CUDA version for engineering optimizations.")
print("The hopper/ subdirectory of FA-3 is excellent material for learning ping-pong scheduling.")


## 4. Triton and Block-Level Programming

Triton is a GPU Kernel DSL with Python-like syntax that compiles to PTX. Its core abstraction is block-level programming: the programmer describes one block's computation, while the compiler maps it to warps and lanes. You do not need to write Triton here; understanding `tl.load`, `tl.dot`, `tl.store`, and one minimal matmul is enough to navigate many production Kernels.


### 4.1 `tl.load`, `tl.store`, and `tl.dot`

A Triton Kernel reads, computes, and writes. `tl.load(ptr, mask, other)` loads a vector or Tile with masked boundary handling. `tl.dot(a,b)` multiplies blocks shaped $[M,K]$ and $[K,N]$ and is the main operation mapped to Tensor Cores. `tl.store(ptr,value,mask)` writes results with the same boundary discipline.

Operations such as `tl.arange`, `tl.exp`, and `tl.maximum` act element-wise on block registers. Because `tl.dot` receives disproportionate hardware throughput, efficient Attention minimizes non-matmul work in its inner loop.


### 4.2 A Minimal Triton Matmul

The following simplified official-style Kernel contains the central block-level pattern: a grid assigns output Tiles, a K loop loads A/B Tiles, `tl.dot` accumulates in FP32, and `tl.store` writes the result. Compare each pointer offset and mask with the matrix coordinates it represents.


## 5. MoE Kernels

MoE Attention can reuse FlashAttention, but its FFN routes each Token to a subset of experts. Launching one cuBLAS GEMM per expert is inefficient because expert matrices may receive only tens or hundreds of Tokens and incur repeated launch overhead. **Grouped GEMM** computes all experts in one launch. We examine its layouts, Triton structure, routing movement, and the extra engineering difficulty relative to dense GEMM.


### 5.1 Grouped GEMM

Given $E$ pairs $(A_e,B_e)$, one launch must compute every $C_e=A_eB_e$. A padded layout uses grid `(expert, max_tokens/BLOCK_M, N/BLOCK_N)` but wastes work on padding. A contiguous variable-length layout concatenates valid Tokens and stores expert boundaries in `cu_seqlens`; CTAs locate their expert from these boundaries. It is harder but avoids padding and is used by DeepGEMM and variable-length FlashAttention.

DeepGEMM's grouped Kernel reuses dense warp specialization and epilogue infrastructure, while adding expert-aware grid mapping, TMA descriptors, and scales.


### 5.2 Grouped GEMM in Triton

The following simplified Kernel extends ordinary Triton matmul with per-expert pointer offsets and a search over `cu_seqlens`. Global M Tiles are mapped back to an expert and local Token range before the same load/dot/store loop runs.


### 5.3 Scatter/Gather for Expert Routing

An MoE forward pass is:

1. router selects top-k experts;
2. scatter dispatches each Token into contiguous expert buffers;
3. grouped GEMM runs expert FFNs;
4. gather weights and combines outputs at original sequence positions.

Scatter/gather are masked memory movement rather than GEMM. Fused MoE Kernels can combine dispatch, two GEMMs, activation, and combine so intermediate tensors remain on chip instead of round-tripping through HBM.


### 5.4 MoE Kernels versus Dense GEMM

MoE introduces three challenges absent from dense GEMM. **Variable lengths** require dynamic boundaries, per-expert scales, and descriptors. **Load imbalance** makes heavily selected experts bottlenecks; training-time router balancing is essential, while launch grids may assign more CTAs to larger experts. **Fusion complexity** requires managing scatter, two GEMMs, activation, and gather with variable lifetimes on chip. CUTLASS `examples/55_hopper_moe` is a useful complete reference.


In [ ]:
# Simulate the MoE scatter data flow with NumPy to understand grouped GEMM inputs
import numpy as np

np.random.seed(42)

# Scenario: eight Tokens, three experts, top-1 routing
n_tokens = 8
n_experts = 3
d = 4

# Expert selected for each Token, produced by the router
routing = np.random.randint(0, n_experts, size=n_tokens)
print(f"Routing decisions: {routing}")
print(f"Hidden-state shape for each Token: [{n_tokens}, {d}]")

# Scatter: group by expert so Tokens for each expert are contiguous
cu_seqlens = [0]
for e in range(n_experts):
    cu_seqlens.append(cu_seqlens[-1] + (routing == e).sum())
print(f"\ncu_seqlens, cumulative boundaries: {cu_seqlens}")
print("Meaning: expert 0 occupies [0,3), expert 1 [3,6), and expert 2 [6,8)")

# Build the contiguous buffer after scatter
scattered = np.zeros((n_tokens, d))
perm = []
for e in range(n_experts):
    for i, r in enumerate(routing):
        if r == e:
            perm.append(i)
perm = np.array(perm)
print(f"\nPermutation, original Token position -> new position: {perm}")
print("Meaning: move original Tokens [2,5,7] into expert 0 positions [0,1,2]")
print("       The router sent these three tokens to expert 0")

print()
print("Key observation:")
print("  After scatter the buffer is contiguous and can feed a grouped GEMM kernel directly")
print("  cu_seqlens gives the kernel each expert's start and end")
print("  Gather reverses scatter, combining results back into original positions with routing weights")


## 6. Learning Path

1. Build hardware intuition: SMs, Tensor Cores, HBM/shared memory, and the Roofline distinction between compute-bound and memory-bound operations.
2. Read Triton tutorials from vector addition through fused Attention.
3. Read FlashAttention's Triton implementation, then cross-reference the FA-2 CUDA loop.
4. Follow the DeepGEMM route in Section 2.4 to learn TMA, WGMMA, and warp specialization.
5. Read CUTLASS Hopper examples for MoE and FMHA.
6. Only then implement and benchmark a custom Kernel, comparing correctness, numerical error, bandwidth, and occupancy.

The guiding principle is block-level intuition first, production source second, and original implementation last.


### 6.1 Recommended Resources

Use resources progressively: the Triton paper and tutorials for block programming; FlashAttention papers and talks for online Softmax, work partitioning, and Hopper scheduling; GPU Mode talks for implementation details; DeepGEMM source for focused Hopper GEMM; and CUTLASS examples for reusable production abstractions.


## Summary

What we learned in this section:

- [ ] The core constraints of advanced kernels are: the asynchronous nature of hardware instructions (TMA / WGMMA), the SRAM capacity ceiling, and the low dynamic range of FP8
- [ ] DeepGEMM uses per-block scaling to squeeze FP8 data into the E4M3 range, with the scale factor layout aligned to the TMA descriptor
- [ ] BF16 output with an FP32 accumulator is the standard for FP8 GEMM; the output stage uses stochastically rounded truncate to avoid bias
- [ ] Warp specialization splits the warps inside a CTA into producer (TMA load) and consumer (WGMMA), synchronized with mbarrier
- [ ] The two FA-2 improvements: fewer non-matmul FLOPs, and parallelism along the seq dimension so that even batch=1 can fill the SMs
- [ ] FA-3 uses ping-pong scheduling to overlap softmax and WGMMA, plus FP8 tensor cores, reaching 75% of H100 peak compute
- [ ] The three core ops of Triton are `tl.load` / `tl.store` / `tl.dot`, where `tl.dot` is the only op that touches the tensor core
- [ ] Block-level programming lets algorithm engineers write block-level logic; the Triton compiler maps it onto warps and lanes on the SM
- [ ] Grouped GEMM packs the GEMMs of E experts into a single kernel launch and uses `cu_seqlens` to handle variable length
- [ ] MoE kernels are hard to write for three reasons: variable length, load imbalance, and fusion complexity (scatter -> GEMM -> activation -> GEMM -> gather)


## Exercises

These exercises build familiarity with DeepGEMM and FlashAttention repository structure. Open the repositories and verify paths yourself.

> You may ask AI for explanations or hints, but avoid asking it to complete the exercise.

**Exercise 1: Identify DeepGEMM File Responsibilities**

Match the listed repository paths to dispatch, launch, device main-loop, specialization, or epilogue responsibilities.

Hint: use the route in Section 2.4.


In [ ]:
# Exercise 1: match DeepGEMM files to responsibilities
files = [
    "csrc/includes/kernels/gemm/kernel.cuh",
    "csrc/includes/kernels/gemm/epilogue.cuh",
    "csrc/includes/kernels/gemm/warp_specializer.cuh",
]
descriptions = [
    "Define producer and consumer warp roles, initializing and waiting on mbarrier",
    "Main kernel template containing the outer K loop and core load, dot, and store logic",
    "Convert FP32 accumulators to BF16 output and apply per-block scale factors",
]

# TODO: reorder descriptions to correspond one-to-one with files
# Hint: kernel.cuh is the main loop, epilogue is output, and warp_specializer assigns warp roles
answer = None  # TODO: enter a list of indices, for example [1, 2, 0]

assert answer is not None, 'Please replace the placeholder before running the assertion.'
assert len(answer) == 3, "The answer should contain three elements"
assert sorted(answer) == [0, 1, 2], "The answer should be a permutation of [0, 1, 2]"

# Correct answers:
#   kernel.cuh          -> main-loop template             -> descriptions[1]
#   epilogue.cuh        → FP32 → BF16 + scale       → descriptions[2]
#   warp_specializer.cuh → producer/consumer + mbarrier → descriptions[0]
expected = [1, 2, 0]
assert answer == expected, (
    "Hint: kernel.cuh is the main loop, including K-block iteration;"
    "epilogue is the FP32-to-BF16 plus scale output stage;"
    "warp_specializer defines producer/consumer mbarrier synchronization"
)

print("Exercise 1 passed:")
for f, i in zip(files, answer):
    print(f"  {f}")
    print(f"    → {descriptions[i]}")
print()
print("You learned the DeepGEMM kernel, epilogue, and warp_specializer trio:")
print("main loop, output, and warp synchronization—the standard layering of a Hopper GEMM kernel.")


**Exercise 2: Identify the key file of FlashAttention-3**

The Hopper-specific code of FlashAttention-3 lives under the `csrc/flash_attn/hopper/` subdirectory. Three files are listed below; pick the one that **contains the ping-pong scheduling main kernel**.

Hint: Refer to Section 3.3. Ping-pong scheduling is the core trick behind FA-3's performance gain, letting softmax overlap with WGMMA.


In [ ]:
# Exercise 2: select the main FA-3 ping-pong scheduling kernel file
candidates = {
    "A": "csrc/flash_attn/hopper/utils.h",
    "B": "csrc/flash_attn/hopper/flash_fwd_kernel.h",
    "C": "csrc/flash_attn/hopper/epilogue_fwd.hpp",
}

print("Interpretation:")
for k, v in candidates.items():
    print(f"  ({k}) {v}")
print()

answer = None  # TODO: enter A / B / C

assert answer in {"A", "B", "C"}, "The answer must be A, B, or C"
assert answer == "B", (
    "Hint: ping-pong scheduling is core forward-kernel logic,"
    "so it belongs in the kernel file rather than utils or epilogue"
)

print(f"Exercise 2 passed: the main FA-3 ping-pong scheduling kernel is")
print(f"  {candidates[answer]}")
print()
print("You learned that FA-3's hopper/ directory separates kernel, epilogue, and utils;")
print("core algorithms such as ping-pong scheduling live in the kernel file.")


**Exercise 3: Identify the key design of grouped GEMM**

Of the three statements below, which one accurately describes the core difference of grouped GEMM relative to dense GEMM?

Hint: Refer to Section 5.1. The core of grouped GEMM is "packing E independent GEMMs into a single launch".


In [ ]:
# Exercise 3: select the core grouped-GEMM design
options = {
    "A": (
        "Grouped GEMM compresses every expert weight to INT4,"
        "trading precision for higher throughput"
    ),
    "B": (
        "Grouped GEMM packages E expert GEMMs into one kernel launch,"
        "adds an expert dimension to the grid, and handles variable lengths with cu_seqlens"
    ),
    "C": (
        "Grouped GEMM concatenates all expert weights into one large matrix,"
        "then computes them with one ordinary cuBLAS GEMM"
    ),
}

print("Interpretation:")
for k, v in options.items():
    print(f"  ({k}) {v}")
print()

answer = None  # TODO: enter A / B / C

assert answer in {"A", "B", "C"}, "The answer must be A, B, or C"
assert answer == "B", (
    "Hint: A is quantization, unrelated to grouping; C is concatenation, still one dense GEMM and unable to handle variable-length experts;"
    "grouped GEMM means E GEMMs in one launch plus cu_seqlens for variable lengths"
)

print(f"Exercise 3 passed: the core grouped-GEMM design is")
print(f"  {options[answer]}")
print()
print("You learned that grouped GEMM batches E small, variable-length GEMMs;")
print("it is neither quantization nor simple concatenation, and requires a variable-length-aware kernel.")
print("DeepGEMM, CUTLASS, and FlashAttention varlen all implement this pattern.")


## References

- DeepSeek-AI, [DeepGEMM repository](https://github.com/deepseek-ai/DeepGEMM), 2025
- Dao et al., [FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awareness](https://arxiv.org/abs/2205.14135), NeurIPS 2022
- Dao, [FlashAttention-2: Faster Attention with Better Parallelism and Work Partitioning](https://arxiv.org/abs/2307.08691), 2023
- Shah et al., [FlashAttention-3: Fast and Accurate Attention with Asynchrony and Low-precision](https://arxiv.org/abs/2407.08608), 2024
- Tillet et al., [Triton: an intermediate language and compiler for tiled neural network computations](https://www.eecs.harvard.edu/~htk/publication/2019-mapl-tillet-kung-cox), MAPL 2019
- Milakov & Gimelshein, [Online normalizer calculation for softmax](https://arxiv.org/abs/1805.02867), 2018
- NVIDIA, [CUTLASS Hopper examples](https://github.com/NVIDIA/cutlass/tree/main/examples), 2024
- [OpenAI Triton tutorials](https://triton-lang.org/main/getting-started/tutorials/index.html)
